# Notebook 01: Huấn Luyện PhoBERT Token Classification (Medical NER)
### Dự Án: Multimodal Medical AI Chatbot (Phiên Bản 3.0.0)
- **Dataset**: `PB3002/ViMedical_Disease` & `Dqdung205/medical-vietnamese-qa` (Hugging Face)
- **Mô hình gốc**: `vinai/phobert-base-v2`
- **Mục tiêu**: Bóc tách thực thể y tế BIO Tagging (`B-SYMPTOM`, `I-SYMPTOM`, `B-VITAL`, `I-VITAL`, `B-DURATION`, `B-RED_FLAG`) với F1-score $\ge 90\%$.

In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Google Colab
!pip install -q transformers datasets seqeval accelerate pyvi torch evaluate

In [ ]:
# 2. Import thư viện & Kiểm tra GPU
import torch
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

In [ ]:
# 3. Định nghĩa Danh mục Nhãn BIO (Medical Tagset)
LABEL_LIST = [
    'O',
    'B-SYMPTOM', 'I-SYMPTOM',
    'B-VITAL', 'I-VITAL',
    'B-DURATION', 'I-DURATION',
    'B-RED_FLAG', 'I-RED_FLAG'
]
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label = {i: l for i, l in enumerate(LABEL_LIST)}
print('Labels:', label2id)

In [ ]:
# 4. Tải dữ liệu mẫu & Tạo Dataset Pipeline
MODEL_CHECKPOINT = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=False)

# Dữ liệu tổng hợp gán nhãn mẫu cho y tế
sample_medical_data = [
    {
        'tokens': ['Tôi', 'bị', 'sốt', 'cao', '39', 'độ', 'kèm', 'đau', 'đầu', '3', 'ngày', 'nay'],
        'ner_tags': [label2id['O'], label2id['O'], label2id['B-SYMPTOM'], label2id['I-SYMPTOM'], label2id['B-VITAL'], label2id['I-VITAL'], label2id['O'], label2id['B-SYMPTOM'], label2id['I-SYMPTOM'], label2id['B-DURATION'], label2id['I-DURATION'], label2id['O']]
    },
    {
        'tokens': ['Bệnh', 'nhân', 'đau', 'ngực', 'dữ', 'dội', 'vã', 'mồ', 'hôi', 'lan', 'ra', 'tay', 'trái'],
        'ner_tags': [label2id['O'], label2id['O'], label2id['B-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['B-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG'], label2id['I-RED_FLAG']]
    }
]

train_dataset = Dataset.from_list(sample_medical_data * 100)
eval_dataset = Dataset.from_list(sample_medical_data * 20)
print('Train size:', len(train_dataset))

In [ ]:
# 5. Tokenize & Căn chỉnh nhãn Subword Tokenization
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples['tokens'], is_split_into_words=True, truncation=True, max_length=128)
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx] if word_idx < len(label) else -100)
            else:
                label_ids.append(label[word_idx] if word_idx < len(label) else -100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_eval = eval_dataset.map(tokenize_and_align_labels, batched=True)

In [ ]:
# 6. Khởi tạo Mô hình & Thiết lập Hàm tính độ đo Seqeval F1
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id
)

seqeval = evaluate.load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

In [ ]:
# 7. Huấn luyện Mô hình với Hugging Face Trainer
training_args = TrainingArguments(
    output_dir='./results_ner',
    eval_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

print('Bắt đầu quá trình huấn luyện PhoBERT Medical NER...')

In [ ]:
# 8. Xuất Model Weights & Tokenizer sẵn sàng triển khai FastAPI
EXPORT_DIR = './models_weights/ner_phobert'
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)
print(f'Đã xuất thành công model weights và tokenizer sang: {EXPORT_DIR}')